# StreamGuard M1 Production Training + CFA Ablation Proof (GPU)

**Purpose**: Run all 3 ablation configs (B/C/D) on Colab GPU, then generate the CFA proof artifact.

**What this proves**: Config C (GGNN + CFA contrastive loss) outperforms Config B (GGNN baseline) by >= 3 F1 points.

**Prerequisites**:
1. HDF5 split files zipped and uploaded to Google Drive
2. This notebook uploaded to Google Colab
3. Runtime set to **GPU** (T4 or better; A100 recommended for speed)

---

## Estimated Timeline

| Step | What | T4 Time | A100 Time |
|------|------|---------|-----------|
| 1 | Setup (mount, clone, deps, unzip) | 5-10 min | 5-10 min |
| 2 | Dry-run validation | 1-2 min | 1 min |
| 3 | Config B: 20 epochs | 5-8 hours | 2-3 hours |
| 4 | Config C: 20 epochs | 5-8 hours | 2-3 hours |
| 5 | Config D: 20 epochs | 5-8 hours | 2-3 hours |
| 6 | Ablation proof + copy to Drive | 10-15 min | 5-10 min |
| **Total** | | **16-26 hours** | **6-10 hours** |

**Tip**: On T4, run one config per session. On A100, all 3 can fit in one session.

## Cell 1: Verify GPU Runtime

In [ ]:
import torch
import sys
import os

if not torch.cuda.is_available():
    raise RuntimeError(
        "NO GPU DETECTED!\n"
        "Go to: Runtime > Change runtime type > Hardware accelerator > GPU (T4)\n"
        "Then restart the runtime and re-run this cell."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

# Quick CUDA test
x = torch.randn(100, 768, device='cuda')
y = x @ x.T
print(f"CUDA tensor test: OK ({y.shape})")
del x, y
torch.cuda.empty_cache()

# Memory info
free_mem = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Free VRAM: {free_mem:.1f} GB")

# Disk space
import shutil
total, used, free = shutil.disk_usage('/content')
print(f"Disk: {free / 1024**3:.1f} GB free of {total / 1024**3:.1f} GB")

if free / 1024**3 < 15:
    print("WARNING: Less than 15 GB free disk. Training data is ~4 GB + checkpoints ~2 GB.")

## Cell 2: Mount Google Drive

Your Drive should contain the HDF5 data zip at:
```
My Drive/StreamGuard/training_data_final.zip
```

**How to prepare this zip locally** (run on your Windows machine):
```bash
cd C:\Users\Vimal Sajan\streamguard
# Zip the final split files (train.h5 + val.h5 + test.h5 + split_stats.json)
python -c "import shutil; shutil.make_archive('training_data_final', 'zip', '.', 'training/data/final')"
```
Then upload `training_data_final.zip` (~3.5 GB) to `My Drive/StreamGuard/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ---- Configure paths ------------------------------------------------
DRIVE_FOLDER = '/content/drive/MyDrive/Streamguard'
DATA_ZIP     = f'{DRIVE_FOLDER}/training_data_final.zip'

# Local working directories (Colab fast SSD)
WORK_DIR     = '/content/streamguard'
DATA_DIR     = f'{WORK_DIR}/training/data/final'
CKPT_DIR     = f'{WORK_DIR}/training/checkpoints'
PROOF_DIR    = f'{WORK_DIR}/proof'

# Verify zip exists
if not os.path.exists(DATA_ZIP):
    raise FileNotFoundError(
        f"Data zip not found at: {DATA_ZIP}\n"
        f"Upload training_data_final.zip to Google Drive > StreamGuard folder first.\n"
        f"See instructions above this cell."
    )

zip_size_mb = os.path.getsize(DATA_ZIP) / 1024**2
print(f"Found data zip: {zip_size_mb:.0f} MB")

## Cell 3: Clone Repo + Install Dependencies

In [ ]:
%%bash
# Clone repo (or pull if already cloned)
if [ ! -d "/content/streamguard" ]; then
    git clone https://github.com/VimalSajanGeorge/streamguard.git /content/streamguard
else
    cd /content/streamguard && git pull
fi
echo "Repo ready at /content/streamguard"

In [ ]:
# Install only what training needs (minimal, targeted)
!pip install -q \
    'torch>=2.2.0' \
    'torch-geometric>=2.4.0' \
    transformers==4.44.0 \
    'scikit-learn>=1.3.0' \
    'h5py>=3.10.0' \
    'mlflow>=2.12.0' \
    'numpy<2.0'

# Verify all imports
import torch, torch_geometric, transformers, sklearn, h5py, mlflow, numpy
print(f"torch:            {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"torch_geometric:  {torch_geometric.__version__}")
print(f"transformers:     {transformers.__version__}")
print(f"scikit-learn:     {sklearn.__version__}")
print(f"h5py:             {h5py.__version__}")
print(f"mlflow:           {mlflow.__version__}")
print(f"numpy:            {numpy.__version__}")

## Cell 4: Unzip Training Data to Local SSD

Extracts to Colab's local disk for fast I/O during training.

In [ ]:
import zipfile
import time

os.makedirs(DATA_DIR, exist_ok=True)

# Check if already extracted
train_h5 = f'{DATA_DIR}/train.h5'
if os.path.exists(train_h5) and os.path.getsize(train_h5) > 3_000_000_000:
    print(f"Training data already extracted. Skipping.")
else:
    print(f"Extracting {DATA_ZIP} ...")
    start = time.time()
    with zipfile.ZipFile(DATA_ZIP, 'r') as z:
        z.extractall(WORK_DIR)
    elapsed = time.time() - start
    print(f"Extracted in {elapsed:.0f}s")

# Verify all 3 splits
for split in ['train.h5', 'val.h5', 'test.h5']:
    fpath = f'{DATA_DIR}/{split}'
    if not os.path.exists(fpath):
        raise FileNotFoundError(f"Missing: {fpath}")
    size_mb = os.path.getsize(fpath) / 1024**2
    print(f"  {split}: {size_mb:.0f} MB")

# Quick HDF5 integrity check
import h5py
for split in ['train.h5', 'val.h5', 'test.h5']:
    fpath = f'{DATA_DIR}/{split}'
    with h5py.File(fpath, 'r') as f:
        n = len(f['metadata']['labels'])
        print(f"  {split}: {n} samples verified")

print("\nAll data files verified.")

## Cell 5: Dry-Run Validation

Quick test with 100 samples, 1 epoch to verify the full pipeline works before committing to 20 epochs.

**What to check**:
- No import errors or crashes
- `CFA_batches > 0` (proves pairs are batched together)
- Loss values are finite (no NaN/Inf)
- GPU is being used

In [ ]:
import sys
sys.path.insert(0, WORK_DIR)

# Clear any cached modules from previous runs
for mod_name in list(sys.modules.keys()):
    if 'training.scripts.model' in mod_name:
        del sys.modules[mod_name]

from training.scripts.model.train import train, ABLATION_CONFIGS, DEFAULT_CONFIG

print("=" * 60)
print("DRY-RUN VALIDATION (Config C, 1 epoch, 100 samples)")
print("=" * 60)

dry_config = dict(DEFAULT_CONFIG)
ablation = ABLATION_CONFIGS['C_plus_cfa']
dry_config.update({
    'use_cfa': True,
    'cpg_components': ablation['cpg_components'],
    'config_name': 'C_plus_cfa',
    'train_h5': f'{DATA_DIR}/train.h5',
    'val_h5': f'{DATA_DIR}/val.h5',
    'epochs': 1,
    'max_samples': 100,
    'checkpoint_dir': f'{CKPT_DIR}/dry_run',
})

dry_f1 = train(dry_config)

print("\n" + "=" * 60)
print("DRY-RUN COMPLETE")
print("=" * 60)

# Verify GPU was used
if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"Peak GPU memory: {peak:.2f} GB")
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

print("Pipeline validated. Ready for production runs.")

## Cell 6: Config B -- Baseline (CodeBERT + GGNN, NO CFA)

This is the **control**. No contrastive loss, standard cross-entropy only.

**Expected**: ~15-25 min/epoch on T4, ~5-10 min/epoch on A100.

**What to watch**:
- `train_loss` (L_CE) decreasing
- `val_F1` improving after epoch 3-5
- `CFA_batches=0` (correct -- CFA is disabled)

In [ ]:
import time
import torch

# Reload modules (in case of runtime restart)
import sys
sys.path.insert(0, WORK_DIR)
for mod_name in list(sys.modules.keys()):
    if 'training.scripts.model' in mod_name:
        del sys.modules[mod_name]

from training.scripts.model.train import train, ABLATION_CONFIGS, DEFAULT_CONFIG

print("=" * 60)
print("CONFIG B: Baseline (GGNN, no CFA) -- 20 epochs")
print("=" * 60)

config_b = dict(DEFAULT_CONFIG)
ablation_b = ABLATION_CONFIGS['B_plus_ggnn']
config_b.update({
    'use_cfa': False,
    'cpg_components': ablation_b['cpg_components'],
    'config_name': 'B_plus_ggnn',
    'train_h5': f'{DATA_DIR}/train.h5',
    'val_h5': f'{DATA_DIR}/val.h5',
    'epochs': 20,
    'checkpoint_dir': CKPT_DIR,
})

start_b = time.time()
best_f1_b = train(config_b)
elapsed_b = time.time() - start_b

print(f"\nConfig B complete in {elapsed_b/3600:.1f} hours")
print(f"Best val F1: {best_f1_b:.4f}")

# Save checkpoint to Drive immediately (guard against disconnect)
import shutil
os.makedirs(f'{DRIVE_FOLDER}/checkpoints', exist_ok=True)
src = f'{CKPT_DIR}/best_model_B_plus_ggnn.pt'
if os.path.exists(src):
    shutil.copy2(src, f'{DRIVE_FOLDER}/checkpoints/best_model_B_plus_ggnn.pt')
    print(f"Checkpoint backed up to Drive.")

if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"Peak GPU memory: {peak:.2f} GB")
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

## Cell 7: Config C -- CFA Proof (CodeBERT + GGNN + L_CFA)

This is the **proof config**. Same architecture as B, plus CFA contrastive loss.

**Expected**: F1 should exceed Config B by >= 3 percentage points.

**What to watch**:
- `train_L_CFA` **decreasing** from ~0.5 to <0.2 over 5-10 epochs
- `CFA_batches > 50%` of total batches
- `val_F1` improving faster than Config B
- `pairwise_acc` increasing (unique to CFA configs)

In [ ]:
import time
import torch

import sys
sys.path.insert(0, WORK_DIR)
for mod_name in list(sys.modules.keys()):
    if 'training.scripts.model' in mod_name:
        del sys.modules[mod_name]

from training.scripts.model.train import train, ABLATION_CONFIGS, DEFAULT_CONFIG

print("=" * 60)
print("CONFIG C: CFA Proof (GGNN + L_CFA) -- 20 epochs")
print("=" * 60)

config_c = dict(DEFAULT_CONFIG)
ablation_c = ABLATION_CONFIGS['C_plus_cfa']
config_c.update({
    'use_cfa': True,
    'cpg_components': ablation_c['cpg_components'],
    'config_name': 'C_plus_cfa',
    'train_h5': f'{DATA_DIR}/train.h5',
    'val_h5': f'{DATA_DIR}/val.h5',
    'epochs': 20,
    'checkpoint_dir': CKPT_DIR,
})

start_c = time.time()
best_f1_c = train(config_c)
elapsed_c = time.time() - start_c

print(f"\nConfig C complete in {elapsed_c/3600:.1f} hours")
print(f"Best val F1: {best_f1_c:.4f}")

# Save checkpoint to Drive immediately
import shutil, os
os.makedirs(f'{DRIVE_FOLDER}/checkpoints', exist_ok=True)
src = f'{CKPT_DIR}/best_model_C_plus_cfa.pt'
if os.path.exists(src):
    shutil.copy2(src, f'{DRIVE_FOLDER}/checkpoints/best_model_C_plus_cfa.pt')
    print(f"Checkpoint backed up to Drive.")

if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"Peak GPU memory: {peak:.2f} GB")
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

## Cell 8: Config D -- TPG Ablation (CodeBERT + GGNN + L_CFA + TPG)

Same as Config C but adds Taint Propagation Graph edges (4th CPG component).

**Expected**: Similar to Config C in M1 (TPG coverage is partial). Difference validates that the TPG edge pathway works without degrading results.

**What to watch**:
- No NaN (TPG edges can be sparse)
- F1 close to Config C

In [ ]:
import time
import torch

import sys
sys.path.insert(0, WORK_DIR)
for mod_name in list(sys.modules.keys()):
    if 'training.scripts.model' in mod_name:
        del sys.modules[mod_name]

from training.scripts.model.train import train, ABLATION_CONFIGS, DEFAULT_CONFIG

print("=" * 60)
print("CONFIG D: TPG Ablation (GGNN + L_CFA + TPG) -- 20 epochs")
print("=" * 60)

config_d = dict(DEFAULT_CONFIG)
ablation_d = ABLATION_CONFIGS['D_plus_tpg']
config_d.update({
    'use_cfa': True,
    'cpg_components': ablation_d['cpg_components'],
    'config_name': 'D_plus_tpg',
    'train_h5': f'{DATA_DIR}/train.h5',
    'val_h5': f'{DATA_DIR}/val.h5',
    'epochs': 20,
    'checkpoint_dir': CKPT_DIR,
})

start_d = time.time()
best_f1_d = train(config_d)
elapsed_d = time.time() - start_d

print(f"\nConfig D complete in {elapsed_d/3600:.1f} hours")
print(f"Best val F1: {best_f1_d:.4f}")

# Save checkpoint to Drive immediately
import shutil, os
os.makedirs(f'{DRIVE_FOLDER}/checkpoints', exist_ok=True)
src = f'{CKPT_DIR}/best_model_D_plus_tpg.pt'
if os.path.exists(src):
    shutil.copy2(src, f'{DRIVE_FOLDER}/checkpoints/best_model_D_plus_tpg.pt')
    print(f"Checkpoint backed up to Drive.")

if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"Peak GPU memory: {peak:.2f} GB")
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

## Cell 9: Restore Checkpoints from Drive (if Runtime Restarted)

**Run this cell ONLY if your Colab runtime disconnected** and you need to restore checkpoints from Drive before running the ablation comparison.

If all 3 configs ran in a single session, skip this cell.

In [ ]:
import os
import shutil

DRIVE_FOLDER = '/content/drive/MyDrive/StreamGuard'
WORK_DIR     = '/content/streamguard'
CKPT_DIR     = f'{WORK_DIR}/training/checkpoints'
DATA_DIR     = f'{WORK_DIR}/training/data/final'

os.makedirs(CKPT_DIR, exist_ok=True)

# Restore checkpoints from Drive
restored = 0
for cfg in ['B_plus_ggnn', 'C_plus_cfa', 'D_plus_tpg']:
    fname = f'best_model_{cfg}.pt'
    src = f'{DRIVE_FOLDER}/checkpoints/{fname}'
    dst = f'{CKPT_DIR}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1024**2
        print(f"Restored {fname} ({size_mb:.0f} MB)")
        restored += 1
    elif os.path.exists(dst):
        print(f"{fname} already exists locally.")
    else:
        print(f"WARNING: {fname} not found on Drive or locally.")

print(f"\nRestored {restored} checkpoint(s).")

# Also re-extract data if needed
if not os.path.exists(f'{DATA_DIR}/train.h5'):
    print("Data not found locally -- re-extract by running Cell 4.")

## Cell 10: Ablation Comparison + CFA Proof

Evaluates all 3 checkpoints on the held-out test set (3,470 samples, 1,273 CFA pairs).

Generates `proof/cfa_proof_result.json` with the proof artifact.

In [ ]:
import sys
import os

WORK_DIR  = '/content/streamguard'
CKPT_DIR  = f'{WORK_DIR}/training/checkpoints'
DATA_DIR  = f'{WORK_DIR}/training/data/final'

sys.path.insert(0, WORK_DIR)
for mod_name in list(sys.modules.keys()):
    if 'training.scripts.model' in mod_name:
        del sys.modules[mod_name]

os.chdir(WORK_DIR)  # so proof/ is created inside repo

from training.scripts.model.ablation_cfa_vs_baseline import run_ablation

print("=" * 60)
print("ABLATION COMPARISON: B vs C vs D")
print("=" * 60)

result = run_ablation(
    test_h5=f'{DATA_DIR}/test.h5',
    checkpoint_dir=CKPT_DIR,
    batch_size=16,  # larger batch for faster eval on GPU
)

print("\n" + "=" * 60)
if result['proof_positive']:
    print("  >>> PROOF POSITIVE: CFA contrastive loss improves F1 <<<")
    print(f"  Config B F1: {result['config_B_f1']:.4f}")
    print(f"  Config C F1: {result['config_C_f1']:.4f}")
    print(f"  Delta:       +{result['cfa_delta']:.4f}")
else:
    print("  PROOF NEGATIVE: CFA delta < 0.03")
    print(f"  Config B F1: {result['config_B_f1']:.4f}")
    print(f"  Config C F1: {result['config_C_f1']:.4f}")
    print(f"  Delta:       {result['cfa_delta']:+.4f}")
    print("  This may indicate insufficient training or data issues.")
    print("  Check: did early stopping trigger too early? NaN in losses?")
print("=" * 60)

## Cell 11: Individual Config Evaluation (Detailed)

Run eval.py on each checkpoint individually for per-CWE breakdown.

In [ ]:
import json
from pathlib import Path

from training.scripts.model.eval import (
    load_model_from_checkpoint,
    build_test_loader,
    evaluate_checkpoint,
    print_metrics,
)
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loader, dataset = build_test_loader(f'{DATA_DIR}/test.h5', batch_size=16)
print(f"Test set: {len(dataset)} samples, device: {device}")

all_results = {}
for cfg_name in ['B_plus_ggnn', 'C_plus_cfa', 'D_plus_tpg']:
    ckpt_path = f'{CKPT_DIR}/best_model_{cfg_name}.pt'
    if not os.path.exists(ckpt_path):
        print(f"Skipping {cfg_name} -- checkpoint not found")
        continue

    model, cfg, ckpt = load_model_from_checkpoint(ckpt_path, device)
    use_cfa = cfg.get('use_cfa', False)
    metrics = evaluate_checkpoint(model, loader, device, use_cfa=use_cfa)
    print_metrics(metrics, cfg_name)
    all_results[cfg_name] = metrics

    # Save individual result JSON
    out_path = f'{WORK_DIR}/proof/eval_{cfg_name}.json'
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    Path(out_path).write_text(json.dumps(metrics, indent=2, default=str))

    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

print(f"\nDetailed results saved to {WORK_DIR}/proof/")

## Cell 12: Copy All Results to Google Drive

Saves checkpoints, proof JSON, and MLflow data to Drive for download.

In [ ]:
import shutil
import os

DRIVE_FOLDER = '/content/drive/MyDrive/StreamGuard'
WORK_DIR     = '/content/streamguard'
CKPT_DIR     = f'{WORK_DIR}/training/checkpoints'

# ---- Copy checkpoints ------------------------------------------------
os.makedirs(f'{DRIVE_FOLDER}/checkpoints', exist_ok=True)
for cfg in ['B_plus_ggnn', 'C_plus_cfa', 'D_plus_tpg']:
    src = f'{CKPT_DIR}/best_model_{cfg}.pt'
    dst = f'{DRIVE_FOLDER}/checkpoints/best_model_{cfg}.pt'
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1024**2
        print(f"Checkpoint {cfg}: {size_mb:.0f} MB")

# ---- Copy proof artifacts ------------------------------------------------
proof_src = f'{WORK_DIR}/proof'
proof_dst = f'{DRIVE_FOLDER}/proof'
if os.path.exists(proof_src):
    if os.path.exists(proof_dst):
        shutil.rmtree(proof_dst)
    shutil.copytree(proof_src, proof_dst)
    print(f"Proof artifacts copied to Drive.")

# ---- Copy MLflow runs ------------------------------------------------
mlruns_src = f'{WORK_DIR}/mlruns'
mlruns_zip = f'{DRIVE_FOLDER}/mlruns_m1_training'
if os.path.exists(mlruns_src):
    shutil.make_archive(mlruns_zip, 'zip', WORK_DIR, 'mlruns')
    size_mb = os.path.getsize(f'{mlruns_zip}.zip') / 1024**2
    print(f"MLflow runs: {size_mb:.0f} MB")

print(f"\nAll results saved to: {DRIVE_FOLDER}")
print("\nFiles on Drive:")
for root, dirs, files in os.walk(DRIVE_FOLDER):
    for f in files:
        fpath = os.path.join(root, f)
        rel = os.path.relpath(fpath, DRIVE_FOLDER)
        size_mb = os.path.getsize(fpath) / 1024**2
        print(f"  {rel}: {size_mb:.1f} MB")

## Cell 13: Summary + Next Steps

In [ ]:
import json
import os
import torch

WORK_DIR = '/content/streamguard'
proof_path = f'{WORK_DIR}/proof/cfa_proof_result.json'

print("=" * 60)
print("  STREAMGUARD M1 PRODUCTION TRAINING -- SUMMARY")
print("=" * 60)

if os.path.exists(proof_path):
    result = json.loads(open(proof_path).read())
    print(f"")
    print(f"  Config B (baseline) F1:  {result['config_B_f1']:.4f}")
    print(f"  Config C (CFA)      F1:  {result['config_C_f1']:.4f}")
    print(f"  Config D (TPG+CFA)  F1:  {result['config_D_f1']:.4f}")
    print(f"  CFA Delta:               +{result['cfa_delta']:.4f}")
    print(f"  Pairwise Accuracy:       {result['pairwise_accuracy']:.4f}")
    print(f"  Worst CWE F1:            {result.get('worst_group_f1', 'N/A')}")
    print(f"")
    print(f"  PROOF POSITIVE: {result['proof_positive']}")
    print(f"")
    if result['proof_positive']:
        print(f"  The CFA contrastive loss (Config C) outperforms the")
        print(f"  baseline (Config B) by +{result['cfa_delta']:.3f} F1 points.")
        print(f"  This validates the core hypothesis of the StreamGuard paper.")
    else:
        print(f"  CFA delta < 0.03. See troubleshooting in the guide doc.")
else:
    print("  Proof file not found. Run Cell 10 first.")

print(f"")
print(f"=" * 60)

if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"Peak GPU memory (session): {peak:.2f} GB")

import shutil
total, used, free = shutil.disk_usage('/content')
print(f"Disk: {used/1024**3:.1f} GB used, {free/1024**3:.1f} GB free")

print(f"\nNext steps:")
print(f"  1. Download proof/cfa_proof_result.json from Drive")
print(f"  2. Download checkpoints from Drive")
print(f"  3. Run eval.py locally on test.h5 for final metrics")
print(f"  4. Post-POC: Add CodeBERT tokenization (M2) for full F1")